# Synthetic gradient and checkpoint test — OCSF-RainNet
**Engineering test only** on randomly generated features and labels. It does not train on rainfall observations, and produces no paper accuracy results.


In [ ]:
import os
os.environ.setdefault("TF_USE_LEGACY_KERAS","1")
import tensorflow as tf,sys,numpy as np,json,datetime,gc,traceback
from pathlib import Path
tf.random.set_seed(42);np.random.seed(42)
input_root=next((p.parent for p in Path("/kaggle/input").rglob("pilot_audit_summary.json")),None)
assert input_root
sys.path.insert(0,str(input_root))
import rainnet_intensity_models as models
models.CHECKPOINT=input_root/"rainnet_binary_best_05.h5"
models.ARCH=input_root
assert models.pretrained_sha256()=="71e0e6d7e0fa9fa6f541c89f1bfcc57d701bd94730e0508c2dbf45c341de575f"
report={"project":"OCSF-RainNet","mode":"SYNTHETIC_ENGINEERING_GRADIENT_AND_RESUME_SMOKE","scientific_status":"NO_REAL_TRAINING_OR_SEGMENTATION_PERFORMANCE","created_utc":datetime.datetime.now(datetime.timezone.utc).isoformat(),"tensorflow":tf.__version__,"gpus":[z.name for z in tf.config.list_physical_devices("GPU")],"tests":[]}
x=tf.random.uniform((2,32,32,3),dtype=tf.float32,seed=123)
y=tf.random.uniform((2,32,32),minval=0,maxval=5,dtype=tf.int32,seed=42)
mask=tf.random.uniform((2,32,32),seed=77)<0.07
y=tf.where(mask,tf.cast(255,tf.int32),y)
loss_fn=models.masked_weighted_ordinal_loss([1,1,1,1,1],0.12)
assert tf.math.is_finite(loss_fn(y,tf.ones((2,32,32,5))/5))



In [ ]:
variants=[("plain",142637),("fusion",148234),("ordinal",142632),("proposed",148217),("unet",485853),("raunet",508249)]
for variant,expected in variants:
    result={"model":variant}
    try:
        tf.keras.backend.clear_session();gc.collect()
        model=models.build_unet_baseline(variant=="raunet") if variant in ("unet","raunet") else models.build_rainnet_variant(variant,pretrained=True,freeze_backbone=False)
        assert model.count_params()==expected
        opt=tf.keras.optimizers.Adam(learning_rate=1e-4)
        with tf.GradientTape() as tape:
            probs=model(x,training=True)
            loss=loss_fn(y,probs)
        gradients=tape.gradient(loss,model.trainable_variables)
        entries=[(g,v) for g,v in zip(gradients,model.trainable_variables) if g is not None]
        assert entries and all(bool(tf.reduce_all(tf.math.is_finite(g))) for g,_ in entries)
        opt.apply_gradients(entries)
        result["updated_variables"]=len(entries)
        result["synthetic_loss"]=float(loss.numpy())
        assert np.isfinite(result["synthetic_loss"])
        expected_out=model(x,training=False).numpy()
        checkpoint=Path("/kaggle/working")/(variant+"_synthetic.weights.h5")
        model.save_weights(str(checkpoint))
        model2=models.build_unet_baseline(variant=="raunet") if variant in ("unet","raunet") else models.build_rainnet_variant(variant,pretrained=True,freeze_backbone=False)
        model2.load_weights(str(checkpoint))
        observed=model2(x,training=False).numpy()
        maxerr=float(np.max(np.abs(observed-expected_out)))
        assert maxerr<1e-5,maxerr
        result["roundtrip_max_abs_diff"]=maxerr
        checkpoint.unlink()
        result["status"]="PASS"
    except Exception as e:
        result["status"]="FAIL";result["error"]=repr(e);result["traceback"]=traceback.format_exc()[-1600:]
    report["tests"].append(result)
    print("SYNTHETIC_TRAINING_TEST",variant,result["status"],result.get("error",""),flush=True)
report["all_pass"]=all(v["status"]=="PASS" for v in report["tests"])
Path("/kaggle/working/OCSF_Gradient_Checkpoint_Engineering_Smoke.json").write_text(json.dumps(report,indent=2),encoding="utf8")
print("ENGINEERING_ONLY_ALL_PASS",report["all_pass"])
assert report["all_pass"],"Synthetic model gradient tests failed; see output report"

